# How do I compress context for an AI agent without preserving duplicate or conflicting memories?

An agent can collect more context than it can safely use. Repeated preferences waste tokens. Corrections compete with older values. Similar memories from another user may look relevant while being completely ineligible.

This notebook builds a transparent context-compression pipeline with Oracle AI Database. It keeps raw evidence and lineage for inspection while allowing only active, scoped, canonical memories into the prompt.

## What You Will Build

You will create a small developer assistant that turns noisy interaction-derived candidates into a compact memory set. The completed pipeline distinguishes exact duplicates, paraphrased duplicates, explicit corrections, unresolved conflicts, temporary instructions, and independent facts. It then assembles prompt-ready context under a configurable budget.

The core path is deterministic and runs without an external model. That makes every decision visible, reproducible, and testable.

## Architecture at a Glance

```text
Noisy interaction events
        |
        v
Structured memory candidates
        |
        +--> scope eligibility
        +--> exact hash comparison
        +--> scoped vector similarity
        +--> key-aware conflict classification
        |
        v
Deterministic resolution policy
        |
        +--> canonical active memories
        +--> duplicate/supersession/conflict lineage
        +--> rejected or quarantined candidates
        |
        v
Budgeted, prompt-ready context
```

Oracle stores the candidate events, canonical state, relationships, embeddings, and run metrics in one transactional system.

### Why this notebook uses direct Oracle primitives

The core path does **not** call OpenAI and does **not** import the `oracleagentmemory` package. The local tutorial embedder is deterministic, and the SQL and Python are deliberately visible so developers can inspect how scope, duplicate detection, conflict policy, lineage, lifecycle state, and token budgeting work.

This does not make Oracle AI Agent Memory unnecessary. The package is the higher-level production path for applications that want managed threads, messages, extraction, durable memories, retrieval, summaries, and context controls backed by Oracle AI Database. This notebook is the lower-level teaching layer: it explains the compression decisions that a memory platform must make and gives developers concrete invariants they can evaluate when adopting a managed memory API.

## Notebook Structure

1. Prepare the local runtime and connect to Oracle Database Free.
2. Create an isolated tutorial schema and seed noisy interactions.
3. Normalize candidates and detect exact and semantic duplicates.
4. Detect conflicts and apply an explicit resolution policy.
5. Assemble canonical context under a budget.
6. Prove scope isolation, correction, quarantine, and persistence.
7. Evaluate the invariants and inspect the stored Oracle state.

## Prerequisites

- Python 3.10 or later with a Jupyter kernel
- A running Oracle Database Free or Oracle AI Database instance with the `VECTOR` data type and `VECTOR_DISTANCE`
- An application user with `CREATE SESSION`, `CREATE TABLE`, and `CREATE INDEX`, plus an appropriate tablespace quota
- `python-oracledb`, `python-dotenv`, `pandas`, `nbformat`, and `ipykernel`

No OpenAI credential is required. The notebook uses a deterministic local vector function so that duplicate, conflict, and lifecycle decisions remain reproducible.


## Local Oracle Database Setup

This tutorial expects the Oracle Database Free container to be running before the first Python cell executes. The listener should be available on `localhost:1521`, and the application user must be able to create tables and indexes in `FREEPDB1`.

If the container is already running, confirm it with `podman ps` and continue to Step 1. A typical local setup looks like this:

```bash
podman login container-registry.oracle.com
podman pull container-registry.oracle.com/database/free:latest
podman volume create oracle-free-data

podman run -d \
  --name oracle-free \
  -p 127.0.0.1:1521:1521 \
  --env-file oracle-free.container.env \
  -v oracle-free-data:/opt/oracle/oradata \
  container-registry.oracle.com/database/free:latest
```

Keep the container environment file separate from the notebook configuration. The notebook loads the `.env` file by default and accepts `CONTEXT_COMPRESSION_ENV_FILE` as an optional path override.

Only these values are required by this notebook:

```env
ORACLE_USER=<application user>
ORACLE_PASSWORD=<password>
ORACLE_DSN=localhost:1521/FREEPDB1
```

The deterministic tutorial embedder is implemented inside the notebook with a fixed dimension of 64, so it does not need an embedding service, an OpenAI key, or an embedding-model environment variable. Other values may remain in the shared `.env` for other notebooks; this tutorial ignores them.


## Step 1 - Prepare the Local Runtime

The first step establishes a known execution environment before any schema or memory data is created. It loads the shared configuration, checks that the required values are present, opens a reusable Oracle connection pool, and verifies that the connected database can execute the vector operation used later in the notebook.

Run both cells in this step before continuing. If either cell fails, fix the environment or database connection first; later errors would otherwise be harder to diagnose because they depend on the same configuration and pool.


### Load configuration and initialize the run

This cell loads the Python dependencies, locates the shared `.env`, and validates the three Oracle connection values without displaying credentials. It also creates a unique `RUN_ID`, which isolates this execution from rows produced by earlier notebook runs.

`CONTEXT_COMPRESSION_ENV_FILE` is available when the file lives somewhere else. The cell reports the path it loaded, but it never prints the username's password or any other secret.

`RUN_ID` is generated once and reused by every later insert and query. If you restart the kernel, begin again from this cell so the in-memory objects and database rows all refer to the same new run.


In [1]:
import hashlib
import json
import math
import os
import re
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import oracledb
import pandas as pd
from dotenv import load_dotenv


def locate_env_file() -> Path:
    configured_path = os.getenv("CONTEXT_COMPRESSION_ENV_FILE")
    candidates = []
    if configured_path:
        candidates.append(Path(configured_path).expanduser())
    candidates.extend([
        Path.cwd().parent / ".env",
        Path.cwd() / ".env",
        Path.home() / "Downloads" / "Local notebook" / ".env",
    ])
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved.is_file():
            return resolved
    checked = ", ".join(str(path.expanduser()) for path in candidates)
    raise FileNotFoundError(f"Could not find the notebook .env file. Checked: {checked}")


ENV_PATH = locate_env_file()
load_dotenv(ENV_PATH, override=False)

ORACLE_USER = os.getenv("ORACLE_USER")
ORACLE_PASSWORD = os.getenv("ORACLE_PASSWORD")
ORACLE_DSN = os.getenv("ORACLE_DSN")
EMBEDDING_DIMENSION = 64
RUN_ID = uuid4().hex[:20]

missing = [name for name, value in {
    "ORACLE_USER": ORACLE_USER,
    "ORACLE_PASSWORD": ORACLE_PASSWORD,
    "ORACLE_DSN": ORACLE_DSN,
}.items() if not value]
if missing:
    raise RuntimeError(f"Missing environment values in {ENV_PATH}: {', '.join(missing)}")

print(f"Configuration loaded from: {ENV_PATH}")
print(f"Runtime ready. Run ID: {RUN_ID}")


Configuration loaded from: /Users/srinidhi/Downloads/Local notebook/.env
Runtime ready. Run ID: ae993503a1fb42adb1f8


### Connect to Oracle and verify vector support

The notebook opens a small connection pool and performs two preflight checks: it reads the database version and calculates the cosine distance between identical vectors. A distance of `0.0` confirms that the vector expression used later in the tutorial is available.

Connection pooling is used even in the tutorial because later steps repeatedly acquire and release connections. This matches normal application behavior more closely than keeping one global connection open for the entire notebook.

The preflight deliberately fails early if the credentials, service name, listener, or vector feature is unavailable. A connection error usually points to the Podman container or `.env`; a SQL error in the vector query indicates that the connected service does not provide the expected Oracle AI Vector Search capability.


In [2]:
pool = oracledb.create_pool(
    user=ORACLE_USER,
    password=ORACLE_PASSWORD,
    dsn=ORACLE_DSN,
    min=1,
    max=4,
    increment=1,
)

with pool.acquire() as connection:
    with connection.cursor() as cursor:
        cursor.execute("select banner_full from v$version where rownum = 1")
        database_banner = cursor.fetchone()[0]
        cursor.execute(
            '''select vector_distance(
                       to_vector(:left_vector, 2, float32),
                       to_vector(:right_vector, 2, float32),
                       cosine
                   )
                 from dual''',
            left_vector="[1.0, 0.0]",
            right_vector="[1.0, 0.0]",
        )
        vector_distance_check = cursor.fetchone()[0]

print(database_banner.splitlines()[0])
print("Vector preflight distance:", vector_distance_check)

Oracle AI Database 26ai Free Release 23.26.1.0.0 - Develop, Learn, and Run for Free
Vector preflight distance: 0.0


> **How to read the result:** The database banner identifies the connected instance. `Vector preflight distance: 0.0` means two identical vectors compared successfully, so the later Oracle vector query can run. If the distance is not zero or the query fails, stop here rather than continuing with partially supported vector behavior. The banner is also useful when troubleshooting service-name mistakes, because it confirms which database the notebook actually reached.

## Step 2 - Create the Oracle Schema

Context compression needs more than a single vector table. This step creates separate structures for raw candidate events, canonical memory state, typed relationships, and per-run metrics so that the prompt can stay compact without losing the evidence behind each decision.

The DDL is safe to rerun because it checks the data dictionary before creating each object. Rows from different executions are separated by `RUN_ID`, allowing developers to compare or inspect several tutorial runs without rebuilding the schema each time.


### Create the tutorial tables and scope indexes

This cell creates the event, canonical-memory, relationship, and run-metric tables only when they are missing. The two relational indexes support the scope-first access pattern used throughout the notebook; existing tables and indexes are left intact.

`CTX_EVENTS` stores what arrived, while `CTX_MEMORIES` stores the resolved state. `CTX_RELATIONSHIPS` connects evidence to canonical values and records transitions such as supersession, and `CTX_RUNS` captures the aggregate result of an execution.

The vector columns use the notebook's fixed 64-dimensional tutorial vectors. If you later replace the local embedder with a production model, change the dimension through a reviewed schema migration rather than allowing application startup to silently recreate or reinterpret existing columns.


In [3]:
TABLE_NAMES = {
    "events": "CTX_EVENTS",
    "memories": "CTX_MEMORIES",
    "relationships": "CTX_RELATIONSHIPS",
    "runs": "CTX_RUNS",
}

DDL_STATEMENTS = {
    "CTX_EVENTS": f''' 
        CREATE TABLE CTX_EVENTS (
            run_id              VARCHAR2(20) NOT NULL,
            event_id            VARCHAR2(40) NOT NULL,
            tenant_id           VARCHAR2(100) NOT NULL,
            user_id             VARCHAR2(100) NOT NULL,
            agent_id            VARCHAR2(100) NOT NULL,
            thread_id           VARCHAR2(100),
            memory_key          VARCHAR2(160) NOT NULL,
            memory_value        VARCHAR2(500) NOT NULL,
            memory_text         CLOB NOT NULL,
            memory_type         VARCHAR2(30) NOT NULL,
            durability          VARCHAR2(20) NOT NULL,
            cardinality         VARCHAR2(20) NOT NULL,
            explicit_correction NUMBER(1) DEFAULT 0 NOT NULL,
            confidence          NUMBER(5,4) NOT NULL,
            content_hash        VARCHAR2(64) NOT NULL,
            embedding           VECTOR({EMBEDDING_DIMENSION}, FLOAT32) NOT NULL,
            processing_status   VARCHAR2(40) DEFAULT 'PENDING' NOT NULL,
            created_at          TIMESTAMP WITH TIME ZONE NOT NULL,
            CONSTRAINT pk_ctx_events PRIMARY KEY (run_id, event_id)
        )
    ''',
    "CTX_MEMORIES": f'''
        CREATE TABLE CTX_MEMORIES (
            run_id          VARCHAR2(20) NOT NULL,
            memory_id       VARCHAR2(40) NOT NULL,
            tenant_id       VARCHAR2(100) NOT NULL,
            user_id         VARCHAR2(100) NOT NULL,
            agent_id        VARCHAR2(100) NOT NULL,
            thread_id       VARCHAR2(100),
            memory_key      VARCHAR2(160) NOT NULL,
            memory_value    VARCHAR2(500) NOT NULL,
            memory_text     CLOB NOT NULL,
            memory_type     VARCHAR2(30) NOT NULL,
            cardinality     VARCHAR2(20) NOT NULL,
            status          VARCHAR2(20) NOT NULL,
            confidence      NUMBER(5,4) NOT NULL,
            relevance       NUMBER(5,4) NOT NULL,
            token_estimate  NUMBER(10) NOT NULL,
            source_event_ids CLOB NOT NULL,
            content_hash    VARCHAR2(64) NOT NULL,
            embedding       VECTOR({EMBEDDING_DIMENSION}, FLOAT32) NOT NULL,
            valid_from      TIMESTAMP WITH TIME ZONE NOT NULL,
            valid_to        TIMESTAMP WITH TIME ZONE,
            updated_at      TIMESTAMP WITH TIME ZONE NOT NULL,
            CONSTRAINT pk_ctx_memories PRIMARY KEY (run_id, memory_id)
        )
    ''',
    "CTX_RELATIONSHIPS": '''
        CREATE TABLE CTX_RELATIONSHIPS (
            run_id            VARCHAR2(20) NOT NULL,
            relationship_id   VARCHAR2(40) NOT NULL,
            from_id           VARCHAR2(40) NOT NULL,
            to_id             VARCHAR2(40) NOT NULL,
            relationship_type VARCHAR2(30) NOT NULL,
            similarity        NUMBER(8,6) NOT NULL,
            created_at        TIMESTAMP WITH TIME ZONE DEFAULT SYSTIMESTAMP NOT NULL,
            CONSTRAINT pk_ctx_relationships PRIMARY KEY (run_id, relationship_id)
        )
    ''',
    "CTX_RUNS": '''
        CREATE TABLE CTX_RUNS (
            run_id            VARCHAR2(20) PRIMARY KEY,
            status            VARCHAR2(30) NOT NULL,
            raw_count         NUMBER(10) NOT NULL,
            active_count      NUMBER(10) NOT NULL,
            tokens_before     NUMBER(10) NOT NULL,
            tokens_after      NUMBER(10) NOT NULL,
            metrics_json      CLOB NOT NULL,
            created_at        TIMESTAMP WITH TIME ZONE DEFAULT SYSTIMESTAMP NOT NULL
        )
    ''',
}

def create_table_if_missing(connection, table_name, ddl):
    with connection.cursor() as cursor:
        cursor.execute(
            "select count(*) from user_tables where table_name = :table_name",
            table_name=table_name.upper(),
        )
        if cursor.fetchone()[0] == 0:
            cursor.execute(ddl)
    connection.commit()


with pool.acquire() as connection:
    for table_name, ddl in DDL_STATEMENTS.items():
        create_table_if_missing(connection, table_name, ddl)

    with connection.cursor() as cursor:
        indexes = {
            "IDX_CTX_EVENTS_SCOPE": "create index IDX_CTX_EVENTS_SCOPE on CTX_EVENTS (run_id, tenant_id, user_id, agent_id, memory_key)",
            "IDX_CTX_MEM_SCOPE": "create index IDX_CTX_MEM_SCOPE on CTX_MEMORIES (run_id, tenant_id, user_id, agent_id, status, memory_key)",
        }
        for index_name, ddl in indexes.items():
            cursor.execute(
                "select count(*) from user_indexes where index_name = :index_name",
                index_name=index_name,
            )
            if cursor.fetchone()[0] == 0:
                cursor.execute(ddl)
    connection.commit()

print("Oracle schema ready:", ", ".join(TABLE_NAMES.values()))

Oracle schema ready: CTX_EVENTS, CTX_MEMORIES, CTX_RELATIONSHIPS, CTX_RUNS


## Step 3 - Seed Noisy Agent Interactions

Real interaction history is rarely clean. Users repeat preferences, rephrase facts, correct earlier decisions, give temporary instructions, and sometimes make statements that conflict without clearly saying which one is current.

This step creates a deliberately small but varied dataset that exercises those situations. Read through the rows before running the cell: the later assertions refer to specific event IDs, scopes, keys, and values, so understanding the examples makes every resolution outcome easier to follow.


### Define deliberately noisy interaction evidence

The sample contains exact and semantic repeats, a correction, a genuine conflict, a temporary instruction, multi-valued facts, and records from other identity scopes. Keeping these cases in one dataset lets the later assertions prove each policy branch.

Most events inherit a shared tenant, user, and agent scope from `BASE_DEFAULTS`; selected rows override those values to test isolation. `cardinality="multi"` allows several technologies to coexist, whereas the default single cardinality means two different values for the same scoped key require correction or conflict handling.

The timestamps are deterministic so resolution order remains stable across executions. The data is synthetic and contains no real credentials, personal information, or production records.


In [4]:
RAW_INTERACTIONS = [
    {"event_id": "evt-01", "memory_text": "I prefer Python for backend services.", "memory_key": "backend_language", "memory_value": "Python", "memory_type": "preference"},
    {"event_id": "evt-02", "memory_text": " I prefer python for backend services! ", "memory_key": "backend_language", "memory_value": "python", "memory_type": "preference"},
    {"event_id": "evt-03", "memory_text": "Python is my preferred backend language.", "memory_key": "backend_language", "memory_value": "Python", "memory_type": "preference"},
    {"event_id": "evt-04", "memory_text": "I prefer TypeScript for frontend applications.", "memory_key": "frontend_language", "memory_value": "TypeScript", "memory_type": "preference"},
    {"event_id": "evt-05", "memory_text": "Deploy the application to UK South.", "memory_key": "deployment_region", "memory_value": "UK South", "memory_type": "decision"},
    {"event_id": "evt-06", "memory_text": "Correction: deploy the application to UK West instead.", "memory_key": "deployment_region", "memory_value": "UK West", "memory_type": "decision", "explicit_correction": True},
    {"event_id": "evt-07", "memory_text": "The approved agent framework is LangGraph.", "memory_key": "agent_framework", "memory_value": "LangGraph", "memory_type": "decision"},
    {"event_id": "evt-08", "memory_text": "The approved agent framework is CrewAI.", "memory_key": "agent_framework", "memory_value": "CrewAI", "memory_type": "decision"},
    {"event_id": "evt-09", "memory_text": "For this response, keep the answer under five lines.", "memory_key": "response_length", "memory_value": "five lines", "memory_type": "instruction", "durability": "temporary", "thread_id": "thread-current"},
    {"event_id": "evt-10", "memory_text": "The project uses Python.", "memory_key": "technology", "memory_value": "Python", "memory_type": "fact", "cardinality": "multi"},
    {"event_id": "evt-11", "memory_text": "The project uses Oracle AI Database.", "memory_key": "technology", "memory_value": "Oracle AI Database", "memory_type": "fact", "cardinality": "multi"},
    {"event_id": "evt-12", "memory_text": "The repository is oracle-ai-developer-hub.", "memory_key": "repository", "memory_value": "oracle-ai-developer-hub", "memory_type": "fact"},
    {"event_id": "evt-13", "memory_text": "I prefer Python for backend services.", "memory_key": "backend_language", "memory_value": "Python", "memory_type": "preference", "user_id": "user-202"},
    {"event_id": "evt-14", "memory_text": "I prefer Python for backend services.", "memory_key": "backend_language", "memory_value": "Python", "memory_type": "preference", "tenant_id": "globex"},
    {"event_id": "evt-15", "memory_text": "Use concise answers in this task.", "memory_key": "response_style", "memory_value": "concise", "memory_type": "instruction", "thread_id": "thread-a"},
    {"event_id": "evt-16", "memory_text": "Use concise answers in this task.", "memory_key": "response_style", "memory_value": "concise", "memory_type": "instruction", "thread_id": "thread-b"},
]

BASE_DEFAULTS = {
    "tenant_id": "acme",
    "user_id": "user-101",
    "agent_id": "dev-agent",
    "thread_id": None,
    "durability": "durable",
    "cardinality": "single",
    "explicit_correction": False,
    "confidence": 0.9,
}

for offset, interaction in enumerate(RAW_INTERACTIONS):
    for key, value in BASE_DEFAULTS.items():
        interaction.setdefault(key, value)
    interaction["created_at"] = f"2026-11-18T10:{offset:02d}:00+00:00"

print(f"Seeded {len(RAW_INTERACTIONS)} deliberately noisy interactions.")
pd.DataFrame(RAW_INTERACTIONS)[["event_id", "memory_key", "memory_value", "tenant_id", "user_id", "thread_id"]]

Seeded 16 deliberately noisy interactions.


,event_id,memory_key,memory_value,tenant_id,user_id,thread_id
0,evt-01,backend_language,Python,acme,user-101,NaN
1,evt-02,backend_language,python,acme,user-101,NaN
2,evt-03,backend_language,Python,acme,user-101,NaN
3,evt-04,frontend_language,TypeScript,acme,user-101,NaN
4,evt-05,deployment_region,UK South,acme,user-101,NaN
5,evt-06,deployment_region,UK West,acme,user-101,NaN
6,evt-07,agent_framework,LangGraph,acme,user-101,NaN
7,evt-08,agent_framework,CrewAI,acme,user-101,NaN
8,evt-09,response_length,five lines,acme,user-101,thread-current
9,evt-10,technology,Python,acme,user-101,NaN


## Step 4 - Normalize Memory Candidates

Before two memories can be compared, the application needs a consistent representation of their text, structured value, identity scope, vector, and estimated context cost. Normalization removes cosmetic differences for comparison while preserving the original wording as evidence.

This step first defines the reusable compression functions, then previews their effect, and finally writes prepared events to Oracle. The important boundary is that an event is still only a candidate at this stage; it is not allowed into agent context until the resolution policy assigns an eligible lifecycle state.


### Define normalization, classification, and resolution rules

This cell contains the core mechanics. It normalizes text, creates reproducible tutorial vectors, compares candidates only inside the same identity scope, assigns typed relationships, resolves lifecycle state, and selects active memories under a context budget.

It defines functions but intentionally produces no output; the following cells exercise the functions with visible assertions.

The deterministic embedder hashes normalized tokens into a fixed vector. It is intentionally simple: its purpose is to make the tutorial repeatable and to exercise Oracle's vector type, not to recommend a production embedding model.

`classify_relationship` first checks scope and structured key identity, then considers durability, normalized values, correction intent, cardinality, and similarity. `resolve_candidates` turns that classification into lifecycle actions and lineage, while `select_context` operates only on the active state that survives those decisions.


In [5]:
# CORE_COMPRESSION_LOGIC
import hashlib
import math
import re
from collections import defaultdict

SCOPE_FIELDS = ("tenant_id", "user_id", "agent_id", "thread_id")
REQUIRED_FIELDS = (
    "event_id", "tenant_id", "user_id", "agent_id", "memory_key",
    "memory_value", "memory_text", "memory_type", "created_at",
)
TOKEN_ALIASES = {
    "preferred": "prefer", "prefers": "prefer", "preference": "prefer",
    "services": "backend", "service": "backend", "applications": "application",
    "uses": "use", "using": "use", "database": "database",
}
STOP_WORDS = {"a", "an", "and", "for", "i", "is", "my", "the", "to"}


def normalize_text(text):
    "Normalize cosmetic differences without changing the underlying claim."
    if text is None:
        return ""
    lowered = str(text).strip().lower()
    lowered = re.sub(r"[^a-z0-9]+", " ", lowered)
    return re.sub(r"\s+", " ", lowered).strip()


def content_hash(text):
    return hashlib.sha256(normalize_text(text).encode("utf-8")).hexdigest()


def _semantic_tokens(text):
    tokens = []
    for token in normalize_text(text).split():
        token = TOKEN_ALIASES.get(token, token)
        if token not in STOP_WORDS:
            tokens.append(token)
    return tokens


def deterministic_embedding(text, dimension=64):
    "Create a reproducible tutorial vector; replace it with a production embedding model."
    vector = [0.0] * dimension
    for token in _semantic_tokens(text):
        digest = hashlib.sha256(token.encode("utf-8")).digest()
        index = int.from_bytes(digest[:4], "big") % dimension
        sign = 1.0 if digest[4] % 2 == 0 else -1.0
        vector[index] += sign
    norm = math.sqrt(sum(value * value for value in vector))
    return [value / norm for value in vector] if norm else vector


def cosine_similarity(left, right):
    if len(left) != len(right):
        raise ValueError("Embedding dimensions must match")
    left_norm = math.sqrt(sum(value * value for value in left))
    right_norm = math.sqrt(sum(value * value for value in right))
    if left_norm == 0.0 or right_norm == 0.0:
        return 0.0
    return sum(a * b for a, b in zip(left, right)) / (left_norm * right_norm)


def same_scope(left, right):
    return all(left.get(field) == right.get(field) for field in SCOPE_FIELDS)


def _validate_candidate(candidate):
    for field in REQUIRED_FIELDS:
        value = candidate.get(field)
        if value is None or (isinstance(value, str) and not value.strip()):
            raise ValueError(f"Candidate field {field} is required")
    confidence = float(candidate.get("confidence", 0.0))
    if not 0.0 <= confidence <= 1.0:
        raise ValueError("Candidate confidence must be between 0 and 1")


def _prepare_candidate(candidate):
    prepared = dict(candidate)
    _validate_candidate(prepared)
    prepared.setdefault("thread_id", None)
    prepared.setdefault("durability", "durable")
    prepared.setdefault("cardinality", "single")
    prepared.setdefault("explicit_correction", False)
    prepared.setdefault("confidence", 0.9)
    prepared["normalized_text"] = normalize_text(prepared["memory_text"])
    prepared["normalized_value"] = normalize_text(prepared["memory_value"])
    prepared["content_hash"] = content_hash(prepared["memory_text"])
    prepared["embedding"] = deterministic_embedding(prepared["memory_text"])
    prepared["token_estimate"] = max(1, math.ceil(len(prepared["memory_text"]) / 4))
    return prepared


def classify_relationship(existing, candidate, semantic_threshold=0.72):
    if not same_scope(existing, candidate):
        return "OUT_OF_SCOPE", 0.0
    if normalize_text(existing.get("memory_key")) != normalize_text(candidate.get("memory_key")):
        return "INDEPENDENT", 0.0
    if candidate.get("durability", "durable") == "temporary":
        return "TEMPORARY", 0.0

    existing_value = normalize_text(existing.get("memory_value"))
    candidate_value = normalize_text(candidate.get("memory_value"))
    existing_embedding = existing.get("embedding") or deterministic_embedding(existing.get("memory_text", ""))
    candidate_embedding = candidate.get("embedding") or deterministic_embedding(candidate.get("memory_text", ""))
    similarity = cosine_similarity(existing_embedding, candidate_embedding)

    if existing_value == candidate_value:
        if content_hash(existing.get("memory_text", "")) == content_hash(candidate.get("memory_text", "")):
            return "EXACT_DUPLICATE", similarity
        if similarity >= semantic_threshold:
            return "SEMANTIC_DUPLICATE", similarity
        return "COMPATIBLE", similarity
    if existing.get("cardinality", "single") == "multi" or candidate.get("cardinality", "single") == "multi":
        return "INDEPENDENT", similarity
    if candidate.get("explicit_correction", False):
        return "CORRECTION", similarity
    return "CONFLICT", similarity


def resolve_candidates(candidates, semantic_threshold=0.72):
    memories = []
    relationships = []
    decisions = []
    seen_event_ids = set()

    prepared_candidates = [_prepare_candidate(candidate) for candidate in candidates]
    prepared_candidates.sort(key=lambda item: (item["created_at"], item["event_id"]))

    def add_relationship(from_id, to_id, relationship_type, similarity):
        relationships.append({
            "relationship_id": f"rel-{len(relationships) + 1:03d}",
            "from_id": from_id,
            "to_id": to_id,
            "relationship_type": relationship_type,
            "similarity": round(float(similarity), 6),
        })

    def create_memory(candidate):
        memory = {
            "memory_id": f"mem-{len(memories) + 1:03d}",
            **{field: candidate.get(field) for field in SCOPE_FIELDS},
            "memory_key": candidate["memory_key"],
            "memory_value": candidate["memory_value"],
            "memory_text": candidate["memory_text"].strip(),
            "memory_type": candidate["memory_type"],
            "cardinality": candidate.get("cardinality", "single"),
            "status": "ACTIVE",
            "confidence": float(candidate.get("confidence", 0.9)),
            "created_at": candidate["created_at"],
            "updated_at": candidate["created_at"],
            "valid_to": None,
            "content_hash": candidate["content_hash"],
            "embedding": candidate["embedding"],
            "token_estimate": candidate["token_estimate"],
            "source_event_ids": [candidate["event_id"]],
            "relevance": 0.5,
        }
        memories.append(memory)
        add_relationship(candidate["event_id"], memory["memory_id"], "DERIVED_FROM", 1.0)
        return memory

    for candidate in prepared_candidates:
        event_id = candidate["event_id"]
        if event_id in seen_event_ids:
            decisions.append({"event_id": event_id, "action": "SKIPPED_REPLAY", "similarity": 1.0, "target_memory_id": None})
            continue
        seen_event_ids.add(event_id)

        if candidate.get("durability") == "temporary":
            decisions.append({"event_id": event_id, "action": "REJECTED_TEMPORARY", "similarity": 0.0, "target_memory_id": None})
            continue

        eligible = [
            memory for memory in memories
            if memory["status"] == "ACTIVE"
            and same_scope(memory, candidate)
            and normalize_text(memory["memory_key"]) == normalize_text(candidate["memory_key"])
        ]

        matched = None
        relationship = "INDEPENDENT"
        similarity = 0.0
        for memory in eligible:
            current_relationship, current_similarity = classify_relationship(
                memory, candidate, semantic_threshold=semantic_threshold
            )
            if current_relationship in {"EXACT_DUPLICATE", "SEMANTIC_DUPLICATE", "COMPATIBLE", "CORRECTION", "CONFLICT"}:
                matched = memory
                relationship = current_relationship
                similarity = current_similarity
                break

        if matched is None or relationship == "INDEPENDENT":
            memory = create_memory(candidate)
            decisions.append({"event_id": event_id, "action": "CREATED", "similarity": 0.0, "target_memory_id": memory["memory_id"]})
        elif relationship in {"EXACT_DUPLICATE", "SEMANTIC_DUPLICATE", "COMPATIBLE"}:
            if event_id not in matched["source_event_ids"]:
                matched["source_event_ids"].append(event_id)
            matched["confidence"] = max(matched["confidence"], float(candidate["confidence"]))
            matched["updated_at"] = candidate["created_at"]
            relation_type = "DUPLICATE_OF" if relationship != "COMPATIBLE" else "SUPPORTS"
            add_relationship(event_id, matched["memory_id"], relation_type, similarity)
            action = {
                "EXACT_DUPLICATE": "MERGED_EXACT_DUPLICATE",
                "SEMANTIC_DUPLICATE": "MERGED_SEMANTIC_DUPLICATE",
                "COMPATIBLE": "MERGED_COMPATIBLE_EVIDENCE",
            }[relationship]
            decisions.append({"event_id": event_id, "action": action, "similarity": round(similarity, 6), "target_memory_id": matched["memory_id"]})
        elif relationship == "CORRECTION":
            matched["status"] = "SUPERSEDED"
            matched["valid_to"] = candidate["created_at"]
            new_memory = create_memory(candidate)
            add_relationship(new_memory["memory_id"], matched["memory_id"], "SUPERSEDES", similarity)
            decisions.append({"event_id": event_id, "action": "SUPERSEDED_OLDER_VALUE", "similarity": round(similarity, 6), "target_memory_id": new_memory["memory_id"]})
        else:
            add_relationship(event_id, matched["memory_id"], "CONFLICTS_WITH", similarity)
            decisions.append({"event_id": event_id, "action": "QUARANTINED_CONFLICT", "similarity": round(similarity, 6), "target_memory_id": matched["memory_id"]})

    return {"memories": memories, "relationships": relationships, "decisions": decisions}


def select_context(memories, token_budget):
    if token_budget <= 0:
        return []
    active = [memory for memory in memories if memory.get("status") == "ACTIVE"]
    for memory in active:
        memory["context_score"] = round(
            0.65 * float(memory.get("relevance", 0.0))
            + 0.35 * float(memory.get("confidence", 0.0)),
            6,
        )
    active.sort(key=lambda item: (-item["context_score"], item.get("updated_at", ""), item["memory_id"]))
    selected = []
    used_tokens = 0
    for memory in active:
        cost = int(memory.get("token_estimate") or max(1, math.ceil(len(memory["memory_text"]) / 4)))
        if used_tokens + cost <= token_budget:
            selected.append(memory)
            used_tokens += cost
    return selected


def format_context(selected_memories):
    grouped = defaultdict(list)
    for memory in selected_memories:
        grouped[memory.get("memory_type", "memory").upper()].append(memory["memory_text"].strip())
    sections = []
    for memory_type in sorted(grouped):
        body = "\n".join(f"- {text}" for text in grouped[memory_type])
        sections.append(f"[{memory_type}]\n{body}")
    return "\n\n".join(sections)


### Inspect normalization before writing data

This preview makes cosmetic normalization, hashing, and approximate token cost visible for the first four candidates. The original text remains unchanged in the event record; normalized values exist only to support deterministic comparison.

Compare the `normalized_text` column for `evt-01` and `evt-02`: punctuation and case disappear, producing the same content hash. `evt-03` keeps different wording, which is why it must be handled by semantic comparison rather than exact hashing.

The token estimate uses a transparent character-based approximation. It is sufficient for demonstrating budget behavior, but a production application should replace it with the tokenizer used by the target model.


In [6]:
NORMALIZED_PREVIEW = [_prepare_candidate(item) for item in RAW_INTERACTIONS[:4]]
pd.DataFrame(NORMALIZED_PREVIEW)[["event_id", "normalized_text", "content_hash", "token_estimate"]]

,event_id,normalized_text,content_hash,token_estimate
0,evt-01,i prefer python for backend services,bfdafe77bfed2bfb40feaccfea9e5bb6406046aadab6c2...,10
1,evt-02,i prefer python for backend services,bfdafe77bfed2bfb40feaccfea9e5bb6406046aadab6c2...,10
2,evt-03,python is my preferred backend language,5f0b596d49024868e26f205385c1c4c28fca8768c9f8e9...,10
3,evt-04,i prefer typescript for frontend applications,3af83a074fd38b1d3498189485a11784d99a543942bec7...,12


### Persist raw candidates as immutable evidence

Every prepared interaction is inserted into `CTX_EVENTS` with its identity scope, structured key, lifecycle hints, content hash, and vector. At this point the events are evidence awaiting a resolution decision, not prompt-ready memories.

The insert includes both relational fields and the vector representation. Relational columns define ownership and lifecycle policy; the vector supports similarity search only after those eligibility boundaries are applied.

The initial `processing_status` is `PENDING`. A later transaction updates it with the resolver's action, which lets developers audit every input even when it never creates an active canonical memory.


In [7]:
PREPARED_EVENTS = [_prepare_candidate(item) for item in RAW_INTERACTIONS]
insert_event_sql = f'''
    INSERT INTO CTX_EVENTS (
        run_id, event_id, tenant_id, user_id, agent_id, thread_id,
        memory_key, memory_value, memory_text, memory_type, durability,
        cardinality, explicit_correction, confidence, content_hash,
        embedding, processing_status, created_at
    ) VALUES (
        :run_id, :event_id, :tenant_id, :user_id, :agent_id, :thread_id,
        :memory_key, :memory_value, :memory_text, :memory_type, :durability,
        :cardinality, :explicit_correction, :confidence, :content_hash,
        TO_VECTOR(:embedding_json, {EMBEDDING_DIMENSION}, FLOAT32),
        'PENDING',
        TO_TIMESTAMP_TZ(:created_at, 'YYYY-MM-DD"T"HH24:MI:SSTZH:TZM')
    )
'''

with pool.acquire() as connection:
    with connection.cursor() as cursor:
        for event in PREPARED_EVENTS:
            cursor.execute(
                insert_event_sql,
                run_id=RUN_ID,
                event_id=event["event_id"],
                tenant_id=event["tenant_id"],
                user_id=event["user_id"],
                agent_id=event["agent_id"],
                thread_id=event.get("thread_id"),
                memory_key=event["memory_key"],
                memory_value=event["memory_value"],
                memory_text=event["memory_text"],
                memory_type=event["memory_type"],
                durability=event["durability"],
                cardinality=event["cardinality"],
                explicit_correction=1 if event["explicit_correction"] else 0,
                confidence=event["confidence"],
                content_hash=event["content_hash"],
                embedding_json=json.dumps(event["embedding"]),
                created_at=event["created_at"],
            )
    connection.commit()

print(f"Persisted {len(PREPARED_EVENTS)} candidate events in Oracle.")

Persisted 16 candidate events in Oracle.


## Step 5 - Detect Exact Duplicates

Exact duplicates are the safest records to collapse. They express the same structured value in the same identity scope and reduce to the same normalized content, even when punctuation, capitalization, or surrounding whitespace differs.

The goal is not to delete the repeated event. Instead, the event should become additional provenance for one canonical memory, ensuring that audit history remains complete while the prompt receives only one copy of the fact.


### Prove exact-duplicate detection

The first two backend-language statements differ only in case, spacing, and punctuation. The assertion verifies that normalization and the structured value identify the second event as an exact repeat rather than another active memory.

The code prepares both events through the same validation and normalization path used by the full resolver. It then asserts the typed relationship instead of merely displaying a score, so an unexpected policy change causes the notebook to stop at the failing example.

The similarity value is still printed for inspection, but exact equality is established by scoped identity, normalized value, and content hash. Vector similarity is not needed to make this case safe.


In [8]:
first = _prepare_candidate(RAW_INTERACTIONS[0])
exact_repeat = _prepare_candidate(RAW_INTERACTIONS[1])
exact_relationship, exact_similarity = classify_relationship(first, exact_repeat)

assert exact_relationship == "EXACT_DUPLICATE"
print({"relationship": exact_relationship, "similarity": round(exact_similarity, 3)})

{'relationship': 'EXACT_DUPLICATE', 'similarity': 1.0}


## Step 6 - Detect Semantic Duplicates

Semantic duplicates express the same fact with different wording, so a normalized hash alone cannot identify them. This step combines structured identity—scope, memory key, and normalized value—with vector similarity to decide whether a paraphrase can support an existing canonical memory.

The comparison is shown twice: first through the Python policy and then through Oracle `VECTOR_DISTANCE`. Keeping both views visible helps developers separate the business decision from the database operation that proposes nearby candidates.


### Prove semantic-duplicate detection in Python

This comparison uses the deterministic tutorial vectors to recognize two differently worded statements with the same scoped key and value. The threshold is intentionally visible so developers can see that similarity supports a decision but does not replace scope or identity checks.

The paraphrase has a different content hash, so it cannot take the exact-duplicate path. Because its scope, key, and normalized value agree with the existing preference, the vector score may be evaluated against the explicit tutorial threshold.

In production, calibrate this threshold separately for the chosen model and memory type. A score that works for full sentences may behave very differently for short identifiers, code names, or multilingual values.


In [9]:
paraphrase = _prepare_candidate(RAW_INTERACTIONS[2])
semantic_relationship, semantic_similarity = classify_relationship(
    first, paraphrase, semantic_threshold=0.55
)

assert semantic_relationship == "SEMANTIC_DUPLICATE"
assert semantic_similarity >= 0.55
print({"relationship": semantic_relationship, "similarity": round(semantic_similarity, 3)})

{'relationship': 'SEMANTIC_DUPLICATE', 'similarity': 0.866}


### Repeat the semantic comparison inside Oracle

The query filters by run, tenant, user, agent, thread, and memory key before calculating vector distance. Its output should include `evt-03`, proving that Oracle can retrieve the scoped paraphrase without ranking records from other users or tenants.

Notice the order of the SQL clauses: relational predicates form the candidate set, and `VECTOR_DISTANCE` ranks rows only inside that set. This prevents a highly similar record owned by another user or tenant from becoming visible to application code as a candidate.

The query returns a small number of nearest rows for inspection. The Python resolver still decides whether a returned neighbor is a duplicate, compatible evidence, or a distinct fact.


In [10]:
oracle_semantic_sql = f'''
    SELECT event_id,
           memory_text,
           VECTOR_DISTANCE(
               embedding,
               TO_VECTOR(:query_vector, {EMBEDDING_DIMENSION}, FLOAT32),
               COSINE
           ) AS distance
      FROM CTX_EVENTS
     WHERE run_id = :run_id
       AND tenant_id = :tenant_id
       AND user_id = :user_id
       AND agent_id = :agent_id
       AND memory_key = :memory_key
       AND event_id <> :event_id
       AND ((thread_id IS NULL AND :thread_id IS NULL) OR thread_id = :thread_id)
     ORDER BY distance
     FETCH FIRST 3 ROWS ONLY
'''

with pool.acquire() as connection:
    with connection.cursor() as cursor:
        rows = cursor.execute(
            oracle_semantic_sql,
            query_vector=json.dumps(first["embedding"]),
            run_id=RUN_ID,
            tenant_id=first["tenant_id"],
            user_id=first["user_id"],
            agent_id=first["agent_id"],
            memory_key=first["memory_key"],
            event_id=first["event_id"],
            thread_id=first.get("thread_id"),
        ).fetchall()

oracle_candidates = [
    {"event_id": row[0], "memory_text": row[1], "distance": round(float(row[2]), 6)}
    for row in rows
]
assert "evt-03" in {item["event_id"] for item in oracle_candidates}
print("Oracle semantic candidate proof passed.")
pd.DataFrame(oracle_candidates)

Oracle semantic candidate proof passed.


,event_id,memory_text,distance
0,evt-02,I prefer python for backend services!,0.000000
1,evt-03,Python is my preferred backend language.,0.133975


How to read the result: `evt-03` should appear among the nearest scoped candidates. Rows from another user or tenant are absent because the relational predicates define eligibility before vector ranking. A smaller distance means a closer vector match. The exact distance is tutorial-model-specific; the important checks are that the known paraphrase is returned and cross-scope rows are not considered.

## Step 7 - Detect Conflicting Memories

Conflicts are often semantically close because they discuss the same subject. Two statements about the approved framework may share most of their words while naming different values, which means a high similarity score cannot safely be treated as permission to merge.

This step proves that structured single-value identity takes precedence over similarity. Because the second statement is not marked as an explicit correction, the policy keeps the disagreement visible rather than guessing which framework is authoritative.


### Prove that similarity does not erase a conflict

The two framework statements share the same single-valued key but disagree on the value. The resolver must classify this as a conflict even though their wording is similar; silently merging them would hide an unresolved decision.

Both events refer to `agent_framework`, use the same scope, and are single-valued, but their normalized values differ. The second event also lacks `explicit_correction=True`, so the policy has no trustworthy signal that it should replace the first one.

The assertion expects `CONFLICT` regardless of the reasonably high similarity score. This is a central lesson of the notebook: semantic closeness identifies related claims, not the lifecycle action that should be applied to them.


In [11]:
framework_a = _prepare_candidate(RAW_INTERACTIONS[6])
framework_b = _prepare_candidate(RAW_INTERACTIONS[7])
conflict_relationship, conflict_similarity = classify_relationship(framework_a, framework_b)

assert conflict_relationship == "CONFLICT"
print({"relationship": conflict_relationship, "similarity": round(conflict_similarity, 3)})

{'relationship': 'CONFLICT', 'similarity': 0.75}


## Step 8 - Resolve Candidates into Canonical Memory

The earlier steps examined individual relationships. This step applies the same rules to the complete event stream in timestamp order and produces the memory state that downstream retrieval can trust.

Every event receives a typed action, and every accepted relationship becomes explicit lineage. The resulting canonical set contains active values for context, inactive historical versions for audit, and quarantined or rejected evidence that remains inspectable without becoming prompt material.


### Resolve every candidate into typed memory state

The resolver processes events in a stable order and returns canonical memories, lineage relationships, and one decision per event. The displayed tables reveal which candidates created state, merged as evidence, superseded an older value, were quarantined, or were rejected as temporary.

Processing is stable because candidates are sorted by timestamp and event ID. Replayed event IDs are skipped, temporary events are rejected, duplicates attach evidence, explicit corrections create a new active version, and unresolved single-value disagreements are quarantined.

Review both displayed tables. The decision table explains what happened to each input event, while the canonical-state table shows the memory versions that exist after those decisions, including source event IDs for provenance.


In [12]:
RESOLUTION = resolve_candidates(RAW_INTERACTIONS, semantic_threshold=0.55)
MEMORIES = RESOLUTION["memories"]
RELATIONSHIPS = RESOLUTION["relationships"]
DECISIONS = RESOLUTION["decisions"]

decision_frame = pd.DataFrame(DECISIONS)
memory_frame = pd.DataFrame(MEMORIES)

print("Decision counts:")
display(decision_frame.groupby("action").size().rename("count").reset_index())
print("Canonical state:")
display(memory_frame[["memory_id", "memory_key", "memory_value", "tenant_id", "user_id", "thread_id", "status", "source_event_ids"]])

Decision counts:


,action,count
0,CREATED,11
1,MERGED_EXACT_DUPLICATE,1
2,MERGED_SEMANTIC_DUPLICATE,1
3,QUARANTINED_CONFLICT,1
4,REJECTED_TEMPORARY,1
5,SUPERSEDED_OLDER_VALUE,1


Canonical state:


,memory_id,memory_key,memory_value,tenant_id,user_id,thread_id,status,source_event_ids
0,mem-001,backend_language,Python,acme,user-101,NaN,ACTIVE,"[evt-01, evt-02, evt-03]"
1,mem-002,frontend_language,TypeScript,acme,user-101,NaN,ACTIVE,[evt-04]
2,mem-003,deployment_region,UK South,acme,user-101,NaN,SUPERSEDED,[evt-05]
3,mem-004,deployment_region,UK West,acme,user-101,NaN,ACTIVE,[evt-06]
4,mem-005,agent_framework,LangGraph,acme,user-101,NaN,ACTIVE,[evt-07]
5,mem-006,technology,Python,acme,user-101,NaN,ACTIVE,[evt-10]
6,mem-007,technology,Oracle AI Database,acme,user-101,NaN,ACTIVE,[evt-11]
7,mem-008,repository,oracle-ai-developer-hub,acme,user-101,NaN,ACTIVE,[evt-12]
8,mem-009,backend_language,Python,acme,user-202,NaN,ACTIVE,[evt-13]
9,mem-010,backend_language,Python,globex,user-101,NaN,ACTIVE,[evt-14]


> **How to read the result:** The decision counts should include one exact merge, one semantic merge, one quarantined conflict, one rejected temporary item, and one supersession. The canonical-state table keeps history visible while giving each scoped single-valued key no more than one active value. Read the canonical table alongside these counts: merged duplicates should appear in `source_event_ids`, the old region should be `SUPERSEDED`, and the disputed CrewAI candidate should not become an active canonical value.

### Persist canonical state and lineage

This cell assigns tutorial relevance scores, records each event decision, inserts active and historical memory versions, and stores typed relationships. The transaction preserves the evidence needed to explain why a particular value is—or is not—eligible for context.

The event-status updates, memory-version inserts, and relationship inserts are committed together. If this pattern is moved into a service, the resolution calculation and database writes should be protected by transaction and concurrency controls around the scoped memory key.

The relevance scores are fixed tutorial values so budgeting remains deterministic. A real application might calculate them from query relevance, memory type, freshness, business priority, or a learned ranking model, but it should record enough information to explain the choice.


In [13]:
RELEVANCE_BY_KEY = {
    "backend_language": 0.96,
    "frontend_language": 0.91,
    "deployment_region": 0.88,
    "technology": 0.82,
    "repository": 0.70,
    "agent_framework": 0.62,
    "response_style": 0.40,
}
for memory in MEMORIES:
    memory["relevance"] = RELEVANCE_BY_KEY.get(memory["memory_key"], 0.50)

insert_memory_sql = f'''
    INSERT INTO CTX_MEMORIES (
        run_id, memory_id, tenant_id, user_id, agent_id, thread_id,
        memory_key, memory_value, memory_text, memory_type, cardinality,
        status, confidence, relevance, token_estimate, source_event_ids,
        content_hash, embedding, valid_from, valid_to, updated_at
    ) VALUES (
        :run_id, :memory_id, :tenant_id, :user_id, :agent_id, :thread_id,
        :memory_key, :memory_value, :memory_text, :memory_type, :cardinality,
        :status, :confidence, :relevance, :token_estimate, :source_event_ids,
        :content_hash, TO_VECTOR(:embedding_json, {EMBEDDING_DIMENSION}, FLOAT32),
        :valid_from, :valid_to, :updated_at
    )
'''

with pool.acquire() as connection:
    with connection.cursor() as cursor:
        for decision in DECISIONS:
            cursor.execute(
                '''UPDATE CTX_EVENTS
                      SET processing_status = :processing_status
                    WHERE run_id = :run_id AND event_id = :event_id''',
                processing_status=decision["action"],
                run_id=RUN_ID,
                event_id=decision["event_id"],
            )

        for memory in MEMORIES:
            cursor.execute(
                insert_memory_sql,
                run_id=RUN_ID,
                memory_id=memory["memory_id"],
                tenant_id=memory["tenant_id"],
                user_id=memory["user_id"],
                agent_id=memory["agent_id"],
                thread_id=memory.get("thread_id"),
                memory_key=memory["memory_key"],
                memory_value=memory["memory_value"],
                memory_text=memory["memory_text"],
                memory_type=memory["memory_type"],
                cardinality=memory["cardinality"],
                status=memory["status"],
                confidence=memory["confidence"],
                relevance=memory["relevance"],
                token_estimate=memory["token_estimate"],
                source_event_ids=json.dumps(memory["source_event_ids"]),
                content_hash=memory["content_hash"],
                embedding_json=json.dumps(memory["embedding"]),
                valid_from=datetime.fromisoformat(memory["created_at"]),
                valid_to=datetime.fromisoformat(memory["valid_to"]) if memory["valid_to"] else None,
                updated_at=datetime.fromisoformat(memory["updated_at"]),
            )

        for relationship in RELATIONSHIPS:
            cursor.execute(
                '''INSERT INTO CTX_RELATIONSHIPS (
                           run_id, relationship_id, from_id, to_id,
                           relationship_type, similarity
                       ) VALUES (
                           :run_id, :relationship_id, :from_id, :to_id,
                           :relationship_type, :similarity
                       )''',
                run_id=RUN_ID,
                **relationship,
            )
    connection.commit()

print(f"Persisted {len(MEMORIES)} canonical memory versions and {len(RELATIONSHIPS)} lineage relationships.")

Persisted 12 canonical memory versions and 16 lineage relationships.


## Step 9 - Assemble Context Within a Budget

Once the application has removed ineligible state, context compression becomes a bounded selection problem. The selector considers only active memories inside the requested identity scope, calculates a stable context score, and admits records until adding another one would exceed the tutorial budget.

Budgeting happens deliberately late in the pipeline. A token limit should choose among valid memories; it must never be used to hide duplicates, stale values, conflicts, or cross-user records that should have been rejected earlier.


### Assemble a bounded context window

Only active memories inside the target tenant, user, agent, and user-level thread scope are eligible. The selector ranks those valid memories and stops before the 45-token tutorial budget, demonstrating that budgeting happens after lifecycle and scope resolution.

`TARGET_SCOPE` requests user-level memory by setting `thread_id` to `None`. Thread-specific records are therefore not silently mixed into the context. The selected memories are formatted into labeled sections so facts, preferences, and decisions remain distinguishable to the downstream agent.

The assertions verify both the hard budget and active-state eligibility. If the budget changes, the number of selected memories may change, but the selector must never admit a superseded, quarantined, temporary, or out-of-scope record.


In [14]:
TARGET_SCOPE = {"tenant_id": "acme", "user_id": "user-101", "agent_id": "dev-agent", "thread_id": None}
ELIGIBLE_MEMORIES = [memory for memory in MEMORIES if same_scope(memory, TARGET_SCOPE)]
TOKEN_BUDGET = 45
SELECTED_MEMORIES = select_context(ELIGIBLE_MEMORIES, TOKEN_BUDGET)
PROMPT_CONTEXT = format_context(SELECTED_MEMORIES)
TOKENS_USED = sum(memory["token_estimate"] for memory in SELECTED_MEMORIES)

assert TOKENS_USED <= TOKEN_BUDGET
assert all(memory["status"] == "ACTIVE" for memory in SELECTED_MEMORIES)
print(f"Selected {len(SELECTED_MEMORIES)} memories using {TOKENS_USED}/{TOKEN_BUDGET} estimated tokens.")
print(PROMPT_CONTEXT)

Selected 4 memories using 42/45 estimated tokens.
[DECISION]
- Correction: deploy the application to UK West instead.

[FACT]
- The project uses Python.

[PREFERENCE]
- I prefer Python for backend services.
- I prefer TypeScript for frontend applications.


How to read the result: The printed context is the material that could be sent to an agent. It contains only active, correctly scoped memories and reports the estimated tokens used against the 45-token limit. The output is intentionally shorter than the eligible raw history. Its safety comes from lifecycle and scope filtering first; the numerical token reduction is the final optimization, not the decision rule.

## Step 10 - Prove Scope Isolation

Agent memory is useful only when retrieval respects ownership. A statement can be textually identical to the user's preference and still be completely ineligible because it belongs to another tenant, user, agent, or thread.

This step treats scope as an admission rule rather than a ranking hint. The assertions confirm that similar records remain stored for their rightful owners but cannot leak into the current user's eligible memory set.


### Test tenant, user, agent, and thread isolation

Identical Python preferences exist in three ownership scopes. These assertions prove that the target context contains only the correct tenant and user, while the stored corpus still retains the other users' independent memories.

The corpus intentionally contains three active backend-language preferences with identical wording but different owners. Counting all three proves they were not incorrectly merged globally; filtering `ELIGIBLE_MEMORIES` proves only the target user's row is eligible for this context.

This distinction matters for both privacy and correctness. Scope filtering should occur in the database query in a production retrieval path, not only in Python after potentially sensitive rows have already been fetched.


In [15]:
python_preferences = [
    memory for memory in MEMORIES
    if memory["memory_key"] == "backend_language" and memory["status"] == "ACTIVE"
]

assert len(python_preferences) == 3
assert {memory["tenant_id"] for memory in python_preferences} == {"acme", "globex"}
assert {memory["user_id"] for memory in python_preferences} == {"user-101", "user-202"}
assert all(memory["tenant_id"] == "acme" and memory["user_id"] == "user-101" for memory in ELIGIBLE_MEMORIES)
print("Scope isolation checks passed for tenant, user, agent, and thread boundaries.")

Scope isolation checks passed for tenant, user, agent, and thread boundaries.


## Step 11 - Correct and Supersede a Memory

An explicit correction is different from an unexplained contradiction. When the user clearly replaces an earlier single-valued decision, the new value should become active and the old value should become historical without being physically deleted.

This lifecycle preserves both correctness and accountability. Retrieval sees only the current region, while an operator can still trace what changed, when it changed, and which event caused the transition.


### Verify correction and supersession history

An explicit correction makes `UK West` current and marks `UK South` as superseded. The old row remains available for audit, but only the new active value can reach prompt assembly.

The expected result is one active `UK West` row and one `UK South` row with `SUPERSEDED` status. The `SUPERSEDES` relationship makes the transition explicit instead of leaving operators to infer chronology from timestamps alone.

Keeping the earlier version does not mean sending it to the agent. Lifecycle predicates exclude it from context while still allowing audit, rollback analysis, and evaluation of how often users reverse previous decisions.


In [16]:
regions = [memory for memory in MEMORIES if memory["memory_key"] == "deployment_region"]
active_region = [memory for memory in regions if memory["status"] == "ACTIVE"]
superseded_region = [memory for memory in regions if memory["status"] == "SUPERSEDED"]

assert [memory["memory_value"] for memory in active_region] == ["UK West"]
assert [memory["memory_value"] for memory in superseded_region] == ["UK South"]
assert "SUPERSEDES" in {item["relationship_type"] for item in RELATIONSHIPS}
print("The explicit correction replaced the active value without deleting its history.")

The explicit correction replaced the active value without deleting its history.


## Step 12 - Quarantine an Unresolved Conflict

Not every disagreement can be resolved automatically. If two scoped, single-valued claims disagree and the newer event does not carry reliable correction intent, selecting either value would introduce unsupported state into the agent's prompt.

Quarantine is the conservative path. It records the disputed candidate and its relationship to the current memory, keeps the existing canonical value stable, and creates a visible point where a review workflow or stronger domain policy can intervene.


### Verify conflict quarantine

The ambiguous CrewAI statement remains traceable in the event and decision history but does not become active memory. This is the safe outcome when the application cannot prove that a contradictory candidate is an explicit correction.

The first framework value remains active because the later contradictory event did not prove correction intent. The CrewAI candidate is represented through the event decision and relationship history, which prevents evidence loss while keeping the prompt internally consistent.

The explicit `CrewAI not in PROMPT_CONTEXT` assertion tests the user-visible safety boundary. A production application could route the quarantined item to review, request clarification, or expire it according to domain policy.


In [17]:
framework_memories = [memory for memory in MEMORIES if memory["memory_key"] == "agent_framework"]
framework_decisions = [decision for decision in DECISIONS if decision["event_id"] in {"evt-07", "evt-08"}]

assert [memory["memory_value"] for memory in framework_memories if memory["status"] == "ACTIVE"] == ["LangGraph"]
assert any(decision["action"] == "QUARANTINED_CONFLICT" for decision in framework_decisions)
assert "CrewAI" not in PROMPT_CONTEXT
print("The ambiguous CrewAI claim was retained as evidence but excluded from active context.")

The ambiguous CrewAI claim was retained as evidence but excluded from active context.


## Step 13 - Recreate the Pool and Prove Persistence

A notebook variable is not durable agent memory. This step closes the connection pool, removes the previously assembled Python objects, creates a fresh pool, and reconstructs the eligible context from rows stored in Oracle.

The test approximates the boundary an application crosses during a restart or deployment. If the same active values and exclusions survive pool recreation, the behavior depends on database state rather than accidental in-process state.


### Recreate the connection pool and read durable state

Closing the pool removes the notebook's live database connections and deletes the previously assembled Python context objects. A new pool then reloads active memories from Oracle and rebuilds the same safe context, proving that the state is durable rather than held only in memory.

The reload query repeats the full identity and active-status predicates before returning memory rows. The formatted context is then rebuilt from those database results, rather than reused from the earlier in-memory list.

The assertions check both presence and absence: the corrected region must survive, while the superseded region and quarantined framework value must remain excluded. Persistence is therefore tested as a behavioral property, not merely as a row-count check.


In [18]:
EXPECTED_PERSISTED_COUNT = sum(
    1 for memory in MEMORIES
    if memory["status"] == "ACTIVE" and same_scope(memory, TARGET_SCOPE)
)

pool.close()
del SELECTED_MEMORIES, ELIGIBLE_MEMORIES, PROMPT_CONTEXT

pool = oracledb.create_pool(
    user=ORACLE_USER,
    password=ORACLE_PASSWORD,
    dsn=ORACLE_DSN,
    min=1,
    max=4,
    increment=1,
)

with pool.acquire() as connection:
    with connection.cursor() as cursor:
        persisted_rows = cursor.execute(
            '''SELECT memory_id,
                      memory_key,
                      memory_value,
                      DBMS_LOB.SUBSTR(memory_text, 4000, 1),
                      memory_type,
                      status,
                      confidence,
                      relevance,
                      token_estimate,
                      DBMS_LOB.SUBSTR(source_event_ids, 4000, 1)
                 FROM CTX_MEMORIES
                WHERE run_id = :run_id
                  AND tenant_id = :tenant_id
                  AND user_id = :user_id
                  AND agent_id = :agent_id
                  AND thread_id IS NULL
                  AND status = 'ACTIVE'
                ORDER BY memory_id''',
            run_id=RUN_ID,
            tenant_id=TARGET_SCOPE["tenant_id"],
            user_id=TARGET_SCOPE["user_id"],
            agent_id=TARGET_SCOPE["agent_id"],
        ).fetchall()

PERSISTED_MEMORIES = [
    {
        "memory_id": row[0],
        "memory_key": row[1],
        "memory_value": row[2],
        "memory_text": row[3],
        "memory_type": row[4],
        "status": row[5],
        "confidence": float(row[6]),
        "relevance": float(row[7]),
        "token_estimate": int(row[8]),
        "source_event_ids": json.loads(row[9]),
    }
    for row in persisted_rows
]
PERSISTED_SELECTED = select_context(PERSISTED_MEMORIES, TOKEN_BUDGET)
PERSISTED_CONTEXT = format_context(PERSISTED_SELECTED)

assert len(PERSISTED_MEMORIES) == EXPECTED_PERSISTED_COUNT
assert "UK West" in PERSISTED_CONTEXT
assert "UK South" not in PERSISTED_CONTEXT
assert "CrewAI" not in PERSISTED_CONTEXT
print("Pool recreation proof passed.")
print(PERSISTED_CONTEXT)

Pool recreation proof passed.
[DECISION]
- Correction: deploy the application to UK West instead.

[FACT]
- The project uses Python.

[PREFERENCE]
- I prefer Python for backend services.
- I prefer TypeScript for frontend applications.


## Step 14 - Evaluate the Compression Pipeline

Individual demonstrations are useful, but a tutorial also needs one consolidated verification gate. This step measures the complete run and asserts the invariants that make the compressed context safe: correct lifecycle actions, provenance, scope isolation, one active value per single-valued key, persistence, and budget compliance.

The metrics are stored in `CTX_RUNS` as part of the database record. In a production system, the same pattern can support regression testing, operational dashboards, threshold calibration, and investigations into unexpected merge or quarantine rates.


### Evaluate the complete compression pipeline

This cell checks the notebook's safety and compression invariants together: duplicates merge, conflicts quarantine, temporary items stay out, corrections retain history, scopes remain isolated, provenance exists, and the selected context fits the budget. It stores the verified metrics in `CTX_RUNS`.

The evaluation also verifies that each single-valued scoped key has at most one active version. This catches a dangerous class of race or policy failures where competing values both look current even though individual insert operations succeeded.

The before-and-after token totals measure eligible raw events against the final selected context. They are not a quality score by themselves; the safety assertions must pass before token savings are considered meaningful.


In [19]:
exact_merged = sum(decision["action"] == "MERGED_EXACT_DUPLICATE" for decision in DECISIONS)
semantic_merged = sum(decision["action"] == "MERGED_SEMANTIC_DUPLICATE" for decision in DECISIONS)
conflicts_quarantined = sum(decision["action"] == "QUARANTINED_CONFLICT" for decision in DECISIONS)
temporary_rejected = sum(decision["action"] == "REJECTED_TEMPORARY" for decision in DECISIONS)
superseded_count = sum(memory["status"] == "SUPERSEDED" for memory in MEMORIES)
active_memories = [memory for memory in MEMORIES if memory["status"] == "ACTIVE"]
eligible_raw_events = [event for event in PREPARED_EVENTS if same_scope(event, TARGET_SCOPE)]
tokens_before = sum(event["token_estimate"] for event in eligible_raw_events)
tokens_after = sum(memory["token_estimate"] for memory in PERSISTED_SELECTED)
active_crewai = sum(
    memory["status"] == "ACTIVE" and normalize_text(memory["memory_value"]) == "crewai"
    for memory in MEMORIES
)

single_value_keys = set()
for memory in active_memories:
    if memory["cardinality"] == "single":
        identity = tuple(memory.get(field) for field in SCOPE_FIELDS) + (normalize_text(memory["memory_key"]),)
        assert identity not in single_value_keys
        single_value_keys.add(identity)

assert exact_merged >= 1
assert semantic_merged >= 1
assert conflicts_quarantined >= 1
assert temporary_rejected >= 1
assert superseded_count >= 1
assert active_crewai == 0
assert len(active_memories) < len(PREPARED_EVENTS)
assert tokens_after < tokens_before
assert all(memory["source_event_ids"] for memory in MEMORIES)
assert all(memory["status"] == "ACTIVE" for memory in PERSISTED_SELECTED)
assert tokens_after <= TOKEN_BUDGET
assert len(PERSISTED_MEMORIES) == EXPECTED_PERSISTED_COUNT

METRICS = {
    "raw_candidates": len(PREPARED_EVENTS),
    "active_canonical_memories": len(active_memories),
    "exact_duplicates_merged": exact_merged,
    "semantic_duplicates_merged": semantic_merged,
    "conflicts_quarantined": conflicts_quarantined,
    "temporary_items_rejected": temporary_rejected,
    "superseded_versions": superseded_count,
    "tokens_before": tokens_before,
    "tokens_after": tokens_after,
}

with pool.acquire() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            '''INSERT INTO CTX_RUNS (
                   run_id, status, raw_count, active_count,
                   tokens_before, tokens_after, metrics_json
               ) VALUES (
                   :run_id, 'VERIFIED', :raw_count, :active_count,
                   :tokens_before, :tokens_after, :metrics_json
               )''',
            run_id=RUN_ID,
            raw_count=METRICS["raw_candidates"],
            active_count=METRICS["active_canonical_memories"],
            tokens_before=METRICS["tokens_before"],
            tokens_after=METRICS["tokens_after"],
            metrics_json=json.dumps(METRICS, sort_keys=True),
        )
    connection.commit()

print("All compression invariants passed.")
print("Compression summary:")
print(f"  Raw candidates: {METRICS['raw_candidates']}")
print(f"  Active canonical memories: {METRICS['active_canonical_memories']}")
print(f"  Exact duplicates merged: {METRICS['exact_duplicates_merged']}")
print(f"  Semantic duplicates merged: {METRICS['semantic_duplicates_merged']}")
print(f"  Conflicts quarantined: {METRICS['conflicts_quarantined']}")
print(f"  Temporary items rejected: {METRICS['temporary_items_rejected']}")
print(f"  Superseded versions: {METRICS['superseded_versions']}")
print(f"  Estimated context tokens: {METRICS['tokens_before']} -> {METRICS['tokens_after']}")
print(f"  Active CrewAI memories: {active_crewai}")

All compression invariants passed.
Compression summary:
  Raw candidates: 16
  Active canonical memories: 11
  Exact duplicates merged: 1
  Semantic duplicates merged: 1
  Conflicts quarantined: 1
  Temporary items rejected: 1
  Superseded versions: 1
  Estimated context tokens: 112 -> 42
  Active CrewAI memories: 0


How to read the result: `All compression invariants passed` is the notebook's main verification gate. The summary should report 16 raw candidates, 11 active canonical memories, two duplicate merges, one quarantined conflict, one rejected temporary item, one superseded version, and a smaller final token count. If any assertion fails, treat that as a policy regression and inspect the earlier decision tables before trusting the token totals. A smaller context is not acceptable when it was produced by a false merge or a scope leak.

## Step 15 - Inspect the Oracle State

The final technical step looks directly at what the notebook wrote. Table counts confirm that events, canonical versions, relationships, and run metrics were persisted, while the memory table shows the active and historical lifecycle states for this `RUN_ID`.

Use this inspection when adapting the tutorial. It is often easier to diagnose a surprising prompt by examining stored scope, status, source-event lineage, and structured keys than by looking only at the final formatted context.


### Inspect the rows stored in Oracle

The final database inspection shows row counts for the current run and the lifecycle state of every canonical memory version. This view should contain active, superseded, and quarantined outcomes without exposing values from another run.

Counts are limited to the current `RUN_ID`, so rows left behind by earlier executions do not distort the result. The detailed memory query includes ownership, lifecycle status, and source event IDs—the fields most useful when explaining why a memory reached or failed to reach the prompt.

After running this cell, you can issue additional read-only SQL from another client using the printed `RUN_ID`. Leave cleanup disabled until you have finished that inspection.


In [20]:
with pool.acquire() as connection:
    table_counts = []
    with connection.cursor() as cursor:
        for table_name in TABLE_NAMES.values():
            cursor.execute(f"SELECT COUNT(*) FROM {table_name} WHERE run_id = :run_id", run_id=RUN_ID)
            table_counts.append({"table_name": table_name, "row_count": cursor.fetchone()[0]})

        memory_rows = cursor.execute(
            '''SELECT memory_id, memory_key, memory_value, status,
                      tenant_id, user_id, agent_id, thread_id,
                      DBMS_LOB.SUBSTR(source_event_ids, 4000, 1)
                 FROM CTX_MEMORIES
                WHERE run_id = :run_id
                ORDER BY memory_id''',
            run_id=RUN_ID,
        ).fetchall()

display(pd.DataFrame(table_counts))
display(pd.DataFrame(
    memory_rows,
    columns=["memory_id", "memory_key", "memory_value", "status", "tenant_id", "user_id", "agent_id", "thread_id", "source_event_ids"],
))
print("Oracle state inspection complete.")

,table_name,row_count
0,CTX_EVENTS,16
1,CTX_MEMORIES,12
2,CTX_RELATIONSHIPS,16
3,CTX_RUNS,1


,memory_id,memory_key,memory_value,status,tenant_id,user_id,agent_id,thread_id,source_event_ids
0,mem-001,backend_language,Python,ACTIVE,acme,user-101,dev-agent,NaN,"[""evt-01"", ""evt-02"", ""evt-03""]"
1,mem-002,frontend_language,TypeScript,ACTIVE,acme,user-101,dev-agent,NaN,"[""evt-04""]"
2,mem-003,deployment_region,UK South,SUPERSEDED,acme,user-101,dev-agent,NaN,"[""evt-05""]"
3,mem-004,deployment_region,UK West,ACTIVE,acme,user-101,dev-agent,NaN,"[""evt-06""]"
4,mem-005,agent_framework,LangGraph,ACTIVE,acme,user-101,dev-agent,NaN,"[""evt-07""]"
5,mem-006,technology,Python,ACTIVE,acme,user-101,dev-agent,NaN,"[""evt-10""]"
6,mem-007,technology,Oracle AI Database,ACTIVE,acme,user-101,dev-agent,NaN,"[""evt-11""]"
7,mem-008,repository,oracle-ai-developer-hub,ACTIVE,acme,user-101,dev-agent,NaN,"[""evt-12""]"
8,mem-009,backend_language,Python,ACTIVE,acme,user-202,dev-agent,NaN,"[""evt-13""]"
9,mem-010,backend_language,Python,ACTIVE,globex,user-101,dev-agent,NaN,"[""evt-14""]"


Oracle state inspection complete.


## What This Notebook Achieved

This notebook did more than shorten a list of messages. It built an inspectable state transition from noisy evidence to active canonical memory:

- Exact and paraphrased duplicates contributed provenance without consuming another context slot.
- An explicit correction superseded the older deployment region while preserving history.
- An ambiguous framework conflict was quarantined rather than guessed into the prompt.
- Temporary instructions stayed out of durable memory.
- Tenant, user, agent, and thread boundaries were applied before similarity.
- A token budget selected only active canonical memories.
- Oracle AI Database persisted the events, vectors, lifecycle state, lineage, and run metrics across a new connection pool.

### Print the outcome summary

This compact summary translates the detailed assertions into the tutorial's final results: candidates processed, canonical memories retained, duplicate slots removed, unsafe states excluded, and estimated tokens saved.

This cell does not introduce a new test; it summarizes values already verified in Step 14. Use it as the compact handoff for the tutorial, then return to the detailed decision tables when a particular count needs explanation.

The summary demonstrates the intended trade-off: the agent receives fewer tokens and one current version of each fact, while Oracle retains the underlying evidence, history, and relationships needed for investigation.


In [21]:
print("Achievement summary")
for label, value in [
    ("Raw candidates processed", METRICS["raw_candidates"]),
    ("Active canonical memories", METRICS["active_canonical_memories"]),
    ("Duplicate candidates merged", exact_merged + semantic_merged),
    ("Conflicts quarantined", conflicts_quarantined),
    ("Temporary candidates rejected", temporary_rejected),
    ("Superseded versions retained for audit", superseded_count),
    ("Estimated tokens removed", tokens_before - tokens_after),
]:
    print(f"- {label}: {value}")

Achievement summary
- Raw candidates processed: 16
- Active canonical memories: 11
- Duplicate candidates merged: 2
- Conflicts quarantined: 1
- Temporary candidates rejected: 1
- Superseded versions retained for audit: 1
- Estimated tokens removed: 70


## Production Considerations

The notebook deliberately makes the core behavior small enough to inspect. A production memory-compression service needs additional controls:

- **Embedding model and vector dimension:** Replace the tutorial hash embedder with an approved semantic embedding model. Keep the database column dimension, model version, and re-embedding process aligned.
- **Similarity calibration:** Tune duplicate thresholds with a labeled dataset from the real workload. A universal threshold is unlikely to work across preferences, decisions, identifiers, and free-form episodes.
- **Conflict policy:** Structured keys make single-valued conflicts visible. For ambiguous natural-language claims, an LLM may propose a relationship, but the application should retain confidence, evidence, and a safe quarantine path.
- **Transactions and concurrency:** Lock or version the canonical key while resolving candidates so two workers cannot create competing active values. Treat the event, relationship, lifecycle update, and run metric as one transaction.
- **Idempotency and replay:** Use stable source event IDs and unique constraints. Reprocessing the same turn must not duplicate provenance or create another canonical memory.
- **Scope and authorization:** Tenant, user, agent, thread, case, and application permissions must define eligibility before vector ranking. Database filtering complements, but does not replace, application authorization.
- **Vector and relational indexes:** Add a vector index after measuring the expected corpus size and recall requirements. Keep selective relational indexes for scope, lifecycle status, and structured memory keys.
- **Real token counting:** Replace the character-based estimate with the tokenizer used by the target model. Reserve space for system instructions, retrieved evidence, tool results, and the model response.
- **Retention and privacy:** Raw evidence and superseded values help explain decisions, but they also extend data retention. Apply consent, redaction, expiration, deletion, and legal-hold policies explicitly.
- **Observability:** Track merge rates, conflict rates, quarantine backlog, compression ratio, context-token use, scope-filter rejections, and corrections that reverse earlier decisions.
- **Evaluation:** Maintain golden examples for duplicates, non-duplicates, conflicts, corrections, multilingual text, short identifiers, empty inputs, and adversarial near-matches. Test answer quality as well as compression ratio.
- **Oracle AI Agent Memory adoption:** Use the `oracleagentmemory` package when the application needs managed threads, messages, extraction, durable-memory search, summaries, pruning, and context assembly. Keep the policies demonstrated here—scope before similarity, explicit lifecycle state, provenance, conflict quarantine, and budget evaluation—as acceptance criteria for the higher-level integration.
- **Schema lifecycle:** Manage DDL and upgrades outside application startup. The tutorial creates missing objects for convenience; production deployments should use reviewed migrations and least-privilege database roles.

### Confirm the production boundary

The preceding guidance describes what must change before this teaching implementation becomes a production service. This marker also confirms that the core tutorial completed without an external model call.

The tutorial intentionally favors visibility over abstraction. It shows the rules directly with Python, SQL, and Oracle vector columns so developers can understand the contract before introducing higher-level memory orchestration.

Before production use, apply the controls listed above—especially transactional key resolution, calibrated embeddings, real tokenization, authorization, retention, observability, and evaluation. The output marker confirms only that the tutorial path required no external model call; it is not a production-readiness certification.


In [22]:
print('Production checklist recorded. The core tutorial required no external model call.')

Production checklist recorded. The core tutorial required no external model call.


## Optional Cleanup

Cleanup remains disabled by default so developers can inspect the stored state after the tutorial.

### Keep or remove this run's tutorial rows

Cleanup is disabled so you can inspect the stored state after execution. Set `CLEANUP_DEMO_TABLES` to `True` and rerun only this cell when you want to delete rows associated with the current `RUN_ID`; the shared table definitions remain in place.

With the default `False` value, the cell closes the pool and leaves the current run available for SQL inspection. Changing the flag to `True` deletes only rows whose `run_id` matches this execution, in relationship-first order to respect the logical dependencies.

The tables and indexes are intentionally retained for future runs. If you want to remove the schema objects themselves, do that separately with a reviewed cleanup script and confirm that no other tutorial run depends on them.


In [23]:
CLEANUP_DEMO_TABLES = False

if CLEANUP_DEMO_TABLES:
    with pool.acquire() as connection:
        with connection.cursor() as cursor:
            for table_name in ["CTX_RELATIONSHIPS", "CTX_MEMORIES", "CTX_EVENTS", "CTX_RUNS"]:
                cursor.execute(f"DELETE FROM {table_name} WHERE run_id = :run_id", run_id=RUN_ID)
        connection.commit()
    print(f"Removed rows for run {RUN_ID}.")
else:
    print(f"Cleanup skipped. Inspect run {RUN_ID} in Oracle.")

pool.close()

Cleanup skipped. Inspect run ae993503a1fb42adb1f8 in Oracle.


## References

- [Oracle AI Vector Search User's Guide](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/)
- [Oracle AI Database documentation](https://docs.oracle.com/en/database/oracle/oracle-database/26/)
- [Oracle AI Agent Memory: About Agent Memory](https://docs.oracle.com/en/database/oracle/agent-memory/26.6/guide/about.html)
- [python-oracledb documentation](https://python-oracledb.readthedocs.io/)
- [Oracle AI Developer Hub notebooks](https://github.com/oracle-devrel/oracle-ai-developer-hub/tree/main/notebooks)